In [3]:
import csv

In [4]:
pipeline_document = "C:\\Users\\g.luppina\\Desktop\\bluereply\\jenkins\\dumper\\pipelines_20260615_081613.csv"

In [5]:
JENKINS_FILE_MAPPING = {
    "Jenkins_docker_in_docker"                        : "Jenkins_be_java",
    "Jenkins_no_docker_new_maven_master"              : "Jenkins_be_java",
    "Jenkinsfile_docker"                              : "Jenkins_be_java",
    "Jenkinsfile_free_bash_stage"                     : "Jenkins_be_java",
    "Jenkinsfile_JDK_7_NO_SONAR"                      : "Jenkins_be_java",
    "Jenkinsfile_JDK_7_PROFILE_JACOCO"                : "Jenkins_be_java",
    "Jenkinsfile_no_clean"                            : "Jenkins_be_java",
    "Jenkinsfile_no_post"                             : "Jenkins_be_java",
    "Jenkinsfile_prova"                               : "Jenkins_be_java",
    "Jenkinsfile_slave_test"                          : "Jenkins_be_java",
    "Jenkinsfile_Smartweb_Jenkinsv2.330"              : "Jenkins_be_java",
    "Jenkinsfile_test"                                : "Jenkins_be_java",
    "Jenkinsfile-Jenkinsv2.330"                       : "Jenkins_be_java",
    "Jenkinsfile-Jenkinsv2.330-JDK7"                  : "Jenkins_be_java",
    "Jenkins_dockerimage_builder"                     : "Jenkins_be_java",
    "Jenkins_no_docker_JDK_11"                        : "Jenkins_be_java",
    "Jenkins_no_docker_no_registry_slave_quarkus"     : "Jenkins_be_quarkus",
    "Jenkins_no_docker_singleDeploy"                  : "Jenkins_be_java",
    "Jenkins_sgw"                                     : "Jenkins_sgw",
    "Jenkinsfile_JDK_7_NO_INSTALL"                    : "Jenkins_be_java",
    "Jenkinsfile_it"                                  : "Jenkins_be_java",
    "Jenkinsfile_public_libs_JDK11"                   : "Jenkins_public_libs",
    "Jenkinsfile_public_plugins"                      : "Jenkins_be_java",
    "Jenkins_no_docker_no_registry_slave_FE"          : "Jenkins_fe",
    "Jenkins_no_docker_JDK_11_slave"                  : "Jenkins_be_java",
    "Jenkins_no_docker_master"                        : "Jenkins_be_java",
    "Jenkinsfile_public_libs_JDK17"                   : "Jenkins_public_libs",
    "Jenkins_artifact_upload"                         : "Jenkins_upload_artifact",
    "Jenkins_upload_helm"                             : "Jenkins_upload_helm",
    "Jenkinsfile_public_libs"                         : "Jenkins_public_libs",
    "Jenkinsfile_Smartweb"                            : "Jenkins_be_java",
    "Jenkins_no_docker_NPM"                           : "Jenkins_be_java",
    "Jenkinsfile_JDK_7"                               : "Jenkins_be_java",
    "Jenkins_no_docker_new_maven_393"                 : "Jenkins_be_java",
    "Jenkins_no_docker_no_qa_no_deploy"               : "Jenkins_be_java",
    "Jenkins_docker_python_app"                       : "Jenkins_python_app",
    "Jenkins_no_docker_slave"                         : "Jenkins_be_java",
    "Jenkins_docker_python_lib"                       : "Jenkins_python_lib",
    "Jenkins_docker_slave_libNPM"                     : "Jenkins_libNPM",
    "Jenkins_no_docker_no_registry_slave_FE_improved" : "Jenkins_fe",
    "Jenkins_docker_slave_quarkus"                    : "Jenkins_be_quarkus",
    "Jenkins_no_docker_no_registry_slave"             : "Jenkins_be_java",
    "Jenkins_upload_docker_image"                     : "Jenkins_upload_docker_image",
    "Jenkins_no_docker_no_registry"                   : "Jenkins_be_java",
    "Jenkins_no_docker_no_registry_slave_FE_improved2": "Jenkins_fe",
    "Jenkins_no_docker"                               : "Jenkins_be_java",
    "Jenkins_docker_slave_jdk17_sb3"                  : "Jenkins_be_sb3",
    "Jenkins_no_docker_new_maven"                     : "Jenkins_be_java",
    "Jenkinsfile"                                     : "Jenkins_be_java",
    "Jenkins_upload_generic_artifact"                 : "Jenkins_upload_generic_artifact",
    "Jenkins_dotnetcore_sast_from_JSON"               : "Jenkins_dotnetcore_QA",
    "Jenkins_generic_sast_from_JSON"                  : "Jenkins_generic_QA",
}


In [6]:
valid_suffixes = ("-master", "-develop", "-release", "-it", "-lab", "-old", "-chart", "-httpd", "upload-snapshot", "upload-release", "artifact-upload", "upload", "upload-chart")
valid_prefixes = ("standalone-", "BATCH")

In [7]:
ignored = set()
with open("res/ignored.csv", newline="", encoding="utf-8") as file:
    for record in csv.DictReader(file):
        ignored.add((record["FOLDER_NAME"], record["PIPELINE_NAME"]))

redmine_issues = {}
with open("res/redmine_application_codes.csv", newline="", encoding="utf-8") as file:
    for record in csv.DictReader(file):
        key = (record["TEAM"], record["COMPONENT"])
        redmine_issues[key] = record["APPLICATION_CODE"].strip()

In [8]:
pipelines = []

with open(pipeline_document, newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for record in reader:
        code = ""
        record["TO_BE_MIGRATED"] = True

        #
        # Validation checks
        #
        if record["IS_STANDARD"] != "True":
            # print("Pipeline out of standard...", f"{record['FOLDER_NAME']}/{record['PIPELINE_NAME']}")
            record["TO_BE_MIGRATED"] = False

        elif record["SCM_JENKINSFILE"] not in JENKINS_FILE_MAPPING:
            # print("Jenkinsfile not known...", f"{record['FOLDER_NAME']}/{record['PIPELINE_NAME']}")
            record["TO_BE_MIGRATED"] = False

        elif not record["PIPELINE_NAME"].startswith(valid_prefixes) and not record["PIPELINE_NAME"].endswith(valid_suffixes):
            # print("Pipeline name is out of standard...", f"{record['FOLDER_NAME']}/{record['PIPELINE_NAME']}")
            record["TO_BE_MIGRATED"] = False

        #
        # Ignored pipelines
        #
        if record["TO_BE_MIGRATED"]:
            for folder_name, pipeline_name in ignored:

                if record["FOLDER_NAME"] == folder_name and record["PIPELINE_NAME"] == pipeline_name:
                    # print("Pipeline is to be ignored...", f"{record['FOLDER_NAME']}/{record['PIPELINE_NAME']}")
                    record["TO_BE_MIGRATED"] = False
                    break

                if record["FOLDER_NAME"] == folder_name and not pipeline_name:
                    # print("Pipeline is to be ignored...", f"{record['FOLDER_NAME']}/{record['PIPELINE_NAME']}")
                    record["TO_BE_MIGRATED"] = False
                    break

        #
        # Enrich only migratable pipelines
        #
        if record["TO_BE_MIGRATED"]:

            for (folder_name, pipeline_name), code in redmine_issues.items():
                if folder_name == record["FOLDER_NAME"] and pipeline_name in record["PIPELINE_NAME"]:
                    code = code
                    break

            record["APPLICATION_CODE"] = code
            record["SCM_JENKINSFILE"]  = JENKINS_FILE_MAPPING[record["SCM_JENKINSFILE"]]
            record["SCM_BRANCH"]       = "*/new-master"
            if record["PIPELINE_NAME"].startswith("fe"):
                record["SCM_JENKINSFILE"] = "Jenkins_fe"

        record["TO_BE_MIGRATED"] = record.pop("TO_BE_MIGRATED")
        pipelines.append(record)

In [9]:
with open("pipelines.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file,  fieldnames=pipelines[0].keys())
    writer.writeheader()
    writer.writerows(pipelines)